<a href="https://colab.research.google.com/github/zeba-png/Big-six-pairs/blob/main/big_six_pairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Six Bank Pairs Trading — Convergence Backtest

**Question:** When one Canadian bank stock pulls away from a closely linked peer, does the gap reliably close — and is it profitable after costs?

**Universe:** RY, TD, BNS, BMO, CM, NA (TSX, CAD, daily)

**How it works, in one line:** find bank pairs that historically move together, wait for an unusual gap, bet on it closing, and stay out of the way when earnings or big news hit one bank.

Run the cells top to bottom. Every rule is in the `CONFIG` cell.

## 0. Pre-registration (fill in and commit to GitHub BEFORE you look at results)

| Item | Frozen choice |
|---|---|
| Hypothesis | Spreads between cointegrated Big Six pairs mean-revert enough to be profitable after costs |
| Development period (allowed to tune) | 2008–2014 |
| Holdout period (no changes allowed) | 2015 → today |
| Entry / exit / stop | \|z\| ≥ 2.0 / z crosses 0 / \|z\| ≥ 3.5 or 20 days |
| Costs | 5 bps per side + 0.5%/yr borrow on short leg |
| Earnings rule | Flat from 2 days before to 2 days after either bank's earnings; earnings & news jumps stripped from the signal |
| Date committed |2026-10-29|

Anything you change after seeing holdout results must be reported as a change.

In [ ]:
!pip install -q yfinance statsmodels

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from itertools import combinations
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import coint

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.grid': True, 'grid.alpha': .3})

## 1. Strategy rules (CONFIG)

| Rule | Setting | Why |
|---|---|---|
| Pair selection | Every January, test all 15 pairs on the previous 3 years; keep up to 3 with cointegration p < 0.05 and half-life 5–60 days | Relationships drift; re-picking yearly uses only past data (no look-ahead) |
| Hedge ratio | OLS of log prices on the same 3 years, fixed for the year | Defines the "fair" relationship |
| Signal | z-score of spread vs its rolling 60-day mean/std | Measures how unusual today's gap is |
| Entry | \|z\| ≥ 2 → buy the cheap bank, short the expensive one | Bet on convergence |
| Exit | z crosses 0 (converged), \|z\| ≥ 3.5 (stop), 20 days (time stop) | Cap losses when the relationship breaks |
| Execution | Signal at today's close, trade at tomorrow's close | Removes look-ahead bias |
| Costs | 5 bps per side, 0.5%/yr borrow on the short leg | Realistic for large-cap TSX names |

**Earnings protection (3 layers):**
1. **Blackout:** no new trades and any open trade is closed from 2 trading days before to 2 days after *either* bank's earnings. Dates are scheduled in advance, so this uses no future information.
2. **Signal neutralization:** on earnings reaction days, a bank's own move is replaced with its peers' average move when building the *signal*. An earnings re-pricing is treated as permanent news, not a gap that should close. Real P&L still uses real prices.
3. **Jump filter:** any day where one bank moves more than 4 standard deviations away from its peers (M&A, scandal, unscheduled news) is treated the same way, closes open trades in that pair, and blocks new entries for 2 days.

In [ ]:
CONFIG = dict(
    tickers=['RY.TO', 'TD.TO', 'BNS.TO', 'BMO.TO', 'CM.TO', 'NA.TO'],
    benchmark='XFN.TO',             # iShares S&P/TSX Capped Financials
    start='2005-01-01', end=None,
    dev_end='2014-12-31',           # tune only on data up to here; after = holdout
    earnings_csv=None,              # optional: CSV with columns ticker,date to override yfinance

    # pair selection
    formation_years=3, pval_max=0.05, half_life_range=(5, 60), max_pairs=3,

    # signal + trading rules
    z_window=60, z_entry=2.0, z_exit=0.0, z_stop=3.5, max_hold=20,
    exec_delay=1,                   # days between signal and trade (1 = trade at next close)

    # costs
    cost_bps=5.0, borrow_annual=0.005,

    # earnings / news protection
    use_blackout=True, blackout_pre=2, blackout_post=2,
    use_neutralize=True, reaction_days=(0, 1),
    use_jump_filter=True, jump_sigma=4.0, jump_window=60, jump_cooldown=2,
)

## 2. Data

In [ ]:
def load_prices(tickers, start, end=None):
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    px = raw['Close'][tickers]
    return px.dropna(how='any')          # keep only days where every bank traded


def load_earnings_dates(tickers, csv_path=None):
    out = {t: pd.DatetimeIndex([]) for t in tickers}
    if csv_path:
        df = pd.read_csv(csv_path, parse_dates=['date'])
        for t, g in df.groupby('ticker'):
            if t in out:
                out[t] = pd.DatetimeIndex(g['date']).normalize().sort_values()
        return out
    for t in tickers:
        try:
            ed = yf.Ticker(t).get_earnings_dates(limit=100)
            if ed is None or len(ed) == 0:
                continue
            idx = pd.DatetimeIndex(ed.index)
            if idx.tz is not None:
                idx = idx.tz_convert('America/Toronto').tz_localize(None)
            out[t] = idx.normalize().unique().sort_values()
        except Exception as e:
            print(f'{t}: could not fetch earnings dates ({e})')
    return out


def earnings_coverage(earn, start):
    rows = []
    for t, d in earn.items():
        rows.append(dict(ticker=t, n_dates=len(d),
                         first=d.min().date() if len(d) else None,
                         last=d.max().date() if len(d) else None))
    cov = pd.DataFrame(rows).set_index('ticker')
    gaps = [t for t, d in earn.items() if len(d) == 0 or d.min() > pd.Timestamp(start) + pd.Timedelta(days=120)]
    if gaps:
        print('NOTE: earnings dates missing before the first listed date for', gaps,
              '\n      The jump filter still catches big earnings moves in that period, '
              'but the pre-earnings blackout cannot. Add dates via CONFIG["earnings_csv"] to close the gap.')
    return cov

In [ ]:
px = load_prices(CONFIG['tickers'], CONFIG['start'], CONFIG['end'])
bench = load_prices([CONFIG['benchmark']], CONFIG['start'], CONFIG['end'])[CONFIG['benchmark']]
bench = bench.reindex(px.index).ffill()
earn = load_earnings_dates(CONFIG['tickers'], CONFIG['earnings_csv'])

print(f'{len(px):,} trading days, {px.index[0].date()} → {px.index[-1].date()}')
earnings_coverage(earn, CONFIG['start'])

## 3. Engine

In [ ]:
def earnings_day_flags(index, earn, tickers):
    # True on the first trading day on/after each earnings date (banks report pre-market)
    f = pd.DataFrame(False, index=index, columns=tickers)
    for t in tickers:
        for d in earn.get(t, []):
            i = index.searchsorted(d)
            if i < len(index) and (index[i] - d).days <= 4:
                f.iat[i, f.columns.get_loc(t)] = True
    return f


def window_flags(day_flags, pre, post):
    out = day_flags.copy()
    for k in range(1, pre + 1):
        out |= day_flags.shift(-k, fill_value=False)
    for k in range(1, post + 1):
        out |= day_flags.shift(k, fill_value=False)
    return out


def peer_mean(lr, exclude=None):
    # average log return of the OTHER banks each day, ignoring flagged days
    clean = lr if exclude is None else lr.where(~exclude)
    tot, cnt = clean.sum(axis=1), clean.notna().sum(axis=1)
    out = {}
    for c in lr.columns:
        oc = cnt - clean[c].notna()
        out[c] = ((tot - clean[c].fillna(0)) / oc).where(oc > 0, 0.0)
    return pd.DataFrame(out)


def jump_flags(lr, sigma, window):
    # big moves vs peers, judged against the PAST window only (shift(1))
    idio = lr - peer_mean(lr)
    sd = idio.rolling(window, min_periods=window // 2).std().shift(1)
    return (idio.abs() > sigma * sd).fillna(False)


def neutralized_logprices(lr, news):
    # replace a bank's return with its peers' average on news days, then rebuild a price path
    adj = lr.where(~news, peer_mean(lr, exclude=news))
    return adj.fillna(0).cumsum()


def half_life(s):
    s = pd.Series(s).dropna()
    ds, lag = s.diff().iloc[1:], s.shift(1).iloc[1:]
    b = np.linalg.lstsq(np.column_stack([np.ones(len(lag)), lag.values]), ds.values, rcond=None)[0][1]
    return -np.log(2) / b if b < 0 else np.inf


def select_pairs(logp, start, end, cfg):
    w = logp.loc[start:end]
    rows = []
    for a, b in combinations(w.columns, 2):
        # Engle-Granger is direction-sensitive: keep the direction with the lower p-value
        y, x, p = min(((a, b, coint(w[a], w[b])[1]), (b, a, coint(w[b], w[a])[1])), key=lambda r: r[2])
        alpha, beta = np.linalg.lstsq(np.column_stack([np.ones(len(w)), w[x].values]), w[y].values, rcond=None)[0]
        rows.append(dict(y=y, x=x, pval=p, beta=beta, half_life=half_life(w[y] - beta * w[x])))
    allp = pd.DataFrame(rows).sort_values('pval').reset_index(drop=True)
    lo, hi = cfg['half_life_range']
    ok = allp[(allp.pval < cfg['pval_max']) & (allp.beta > 0) & allp.half_life.between(lo, hi)]
    return allp, ok.head(cfg['max_pairs'])


def simulate_sleeve(z, block, jump_day, cool, days, cfg):
    # state machine for one pair in one year. +1 = long spread (buy y, short x), -1 = short spread
    target = pd.Series(0.0, index=z.index)
    trades, pos, held, armed, entry = [], 0, 0, True, None

    def close(d, zt, reason):
        trades.append(dict(entry_date=entry[0], entry_z=entry[1], direction=pos,
                           exit_date=d, exit_z=zt, reason=reason, days=held))

    for d in days:
        zt = z.at[d]
        if pos != 0:
            held += 1
            reason = None
            if block.at[d]:                                   reason = 'earnings blackout'
            elif jump_day.at[d]:                              reason = 'news jump'
            elif np.isnan(zt):                                reason = 'no signal'
            elif abs(zt) >= cfg['z_stop']:                    reason = 'stop loss'
            elif (pos == 1 and zt >= -cfg['z_exit']) or (pos == -1 and zt <= cfg['z_exit']):
                                                              reason = 'converged'
            elif held >= cfg['max_hold']:                     reason = 'time stop'
            if reason:
                close(d, zt, reason); pos, armed = 0, False
        if pos == 0 and not np.isnan(zt):
            if not armed and abs(zt) < cfg['z_entry']:
                armed = True                                  # re-arm only after z resets
            if armed and not block.at[d] and not cool.at[d] and cfg['z_entry'] <= abs(zt) < cfg['z_stop']:
                pos, held, entry = (-1 if zt > 0 else 1), 0, (d, zt)
        target.at[d] = pos
    if pos != 0:                                              # flatten at year end, pairs are re-picked
        target.at[days[-1]] = 0
        close(days[-1], z.at[days[-1]], 'year-end rebalance')
    return target, trades


def run_backtest(px, earn, cfg, verbose=False):
    tickers, idx = list(px.columns), px.index
    lr = np.log(px).diff()
    r = px.pct_change().fillna(0.0)
    none = pd.DataFrame(False, index=idx, columns=tickers)

    eday = earnings_day_flags(idx, earn, tickers)
    blackout = window_flags(eday, cfg['blackout_pre'], cfg['blackout_post']) if cfg['use_blackout'] else none
    reaction = eday.copy()
    for k in cfg['reaction_days']:
        if k > 0:
            reaction |= eday.shift(k, fill_value=False)
    jumps = jump_flags(lr, cfg['jump_sigma'], cfg['jump_window']) if cfg['use_jump_filter'] else none
    jump_cool = window_flags(jumps, 0, cfg['jump_cooldown'])

    if cfg['use_neutralize']:
        logp = neutralized_logprices(lr, reaction | jumps)
    else:
        logp = np.log(px) - np.log(px.iloc[0])

    D, K = cfg['exec_delay'], cfg['max_pairs']
    first_year = idx[0].year + cfg['formation_years']
    sleeve_rets, trades, picks = [], [], []

    for Y in range(first_year, idx[-1].year + 1):
        days = idx[idx.year == Y]
        if len(days) == 0:
            continue
        allp, sel = select_pairs(logp, f'{Y - cfg["formation_years"]}-01-01', f'{Y - 1}-12-31', cfg)
        for _, p in sel.iterrows():
            picks.append(dict(year=Y, pair=f"{p.y}/{p.x}", pval=p.pval, beta=p.beta, half_life=p.half_life))
        if verbose:
            print(Y, 'pairs:', ', '.join(f"{p.y}/{p.x}" for _, p in sel.iterrows()) or 'none')

        for _, p in sel.iterrows():
            y, x, beta = p.y, p.x, p.beta
            s = logp[y] - beta * logp[x]
            z = (s - s.rolling(cfg['z_window']).mean()) / s.rolling(cfg['z_window']).std()
            block = blackout[y] | blackout[x]
            jday = jumps[y] | jumps[x]
            cool = jump_cool[y] | jump_cool[x]
            target, tr = simulate_sleeve(z, block, jday, cool, days, cfg)

            wy, wx = 1 / (1 + beta), -beta / (1 + beta)            # dollar weights per unit of spread
            held = target.shift(D + 1).fillna(0)                   # position earning today's return
            traded = target.shift(D).fillna(0).diff().abs().fillna(0)
            gross = held * (wy * r[y] + wx * r[x]) / K
            cost = traded * (cfg['cost_bps'] / 1e4) / K
            short_w = np.where(held > 0, abs(wx), np.where(held < 0, wy, 0.0))
            borrow = pd.Series(short_w, index=idx) * cfg['borrow_annual'] / 252 / K
            net = gross - cost - borrow
            sleeve_rets.append(pd.DataFrame({'gross': gross, 'net': net}))

            for t in tr:                                            # P&L per trade, from entry fill to exit fill
                i0, i1 = idx.get_loc(t['entry_date']) + D, idx.get_loc(t['exit_date']) + D
                i1 = min(i1, len(idx) - 1)
                t.update(pair=f'{y}/{x}', pnl=net.iloc[i0:i1 + 1].sum() * K)   # P&L on the sleeve's own capital
                trades.append(t)

    tot = sum(sleeve_rets) if sleeve_rets else pd.DataFrame({'gross': 0.0, 'net': 0.0}, index=idx)
    start = pd.Timestamp(f'{first_year}-01-01')
    return dict(gross=tot['gross'].loc[start:], net=tot['net'].loc[start:],
                trades=pd.DataFrame(trades), picks=pd.DataFrame(picks),
                eday=eday, jumps=jumps, logp=logp)

## 4. Metrics

In [ ]:
def perf_stats(r, bench_r=None):
    r = r.dropna()
    if len(r) < 2 or r.std() == 0:
        return pd.Series(dtype=float)
    eq = (1 + r).cumprod()
    yrs = len(r) / 252
    out = {
        'CAGR': eq.iloc[-1] ** (1 / yrs) - 1,
        'Volatility': r.std() * np.sqrt(252),
        'Sharpe': r.mean() / r.std() * np.sqrt(252),
        'Max drawdown': (eq / eq.cummax() - 1).min(),
        '% days invested': (r != 0).mean(),
    }
    out['Calmar'] = out['CAGR'] / abs(out['Max drawdown']) if out['Max drawdown'] < 0 else np.nan
    if bench_r is not None:
        b = bench_r.reindex(r.index).fillna(0)
        out['Correlation to XFN'] = r.corr(b)
        out['Beta to XFN'] = r.cov(b) / b.var()
    return pd.Series(out)


def trade_stats(tr):
    if tr is None or len(tr) == 0:
        return pd.Series(dtype=float)
    return pd.Series({'Trades': len(tr), 'Win rate': (tr.pnl > 0).mean(),
                      'Avg P&L / trade': tr.pnl.mean(), 'Avg days held': tr.days.mean(),
                      'Trades / year': len(tr) / max(tr.entry_date.dt.year.nunique(), 1)})


def split_report(res, bench, cfg):
    dev_end = pd.Timestamp(cfg['dev_end'])
    br = bench.pct_change()
    periods = {'Development': (None, dev_end), 'Holdout': (dev_end + pd.Timedelta(days=1), None), 'Full': (None, None)}
    rows = {}
    for name, (a, b) in periods.items():
        n = res['net'].loc[a:b]
        g = res['gross'].loc[a:b]
        tr = res['trades']
        if len(tr):
            tr = tr[(tr.entry_date >= (a or tr.entry_date.min())) & (tr.entry_date <= (b or tr.entry_date.max()))]
        s = perf_stats(n, br)
        s['Sharpe (before costs)'] = perf_stats(g).get('Sharpe', np.nan)
        rows[name] = pd.concat([s, trade_stats(tr)])
    return pd.DataFrame(rows)

## 5. Run the strategy

In [ ]:
res = run_backtest(px, earn, CONFIG, verbose=True)
summary = split_report(res, bench, CONFIG)
summary

In [ ]:
print('Exit reasons:')
display(res['trades'].reason.value_counts().to_frame('count'))
print('P&L by pair (sum of trade returns on sleeve capital):')
display(res['trades'].groupby('pair').pnl.agg(['count', 'sum', 'mean']).sort_values('sum'))
print('Pairs selected each year:')
display(res['picks'])

## 6. Charts

In [ ]:
dev_end = pd.Timestamp(CONFIG['dev_end'])
eq_net, eq_gross = (1 + res['net']).cumprod(), (1 + res['gross']).cumprod()
eq_b = (1 + bench.pct_change().reindex(eq_net.index).fillna(0)).cumprod()

fig, ax = plt.subplots()
ax.plot(eq_net, label='Strategy (after costs)', lw=1.6)
ax.plot(eq_gross, label='Strategy (before costs)', lw=1, alpha=.7)
ax.plot(eq_b, label='XFN buy & hold', lw=1, color='grey', alpha=.7)
ax.axvspan(dev_end, eq_net.index[-1], color='orange', alpha=.08, label='Holdout')
ax.set_title('Growth of $1'); ax.legend(); plt.show()

dd = eq_net / eq_net.cummax() - 1
fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_between(dd.index, dd, 0, color='firebrick', alpha=.5); ax.set_title('Strategy drawdown'); plt.show()

ann = res['net'].groupby(res['net'].index.year).apply(lambda r: (1 + r).prod() - 1)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(ann.index, ann.values, color=['seagreen' if v > 0 else 'firebrick' for v in ann.values])
ax.axvline(dev_end.year + .5, color='orange', ls='--'); ax.set_title('Annual return (after costs)'); plt.show()

In [ ]:
# Zoom into one pair-year: spread z-score, trades, and earnings days
def plot_pair_year(res, pair, year, cfg):
    y, x = pair.split('/')
    pk = res['picks'].query('year == @year and pair == @pair')
    if pk.empty:
        print('That pair was not traded that year. Options:'); display(res['picks'].query('year == @year')); return
    beta = pk.beta.iloc[0]
    s = res['logp'][y] - beta * res['logp'][x]
    z = ((s - s.rolling(cfg['z_window']).mean()) / s.rolling(cfg['z_window']).std()).loc[str(year)]
    fig, ax = plt.subplots()
    ax.plot(z, lw=1.2, label='z-score')
    for lvl, c in [(cfg['z_entry'], 'green'), (cfg['z_stop'], 'red')]:
        ax.axhline(lvl, color=c, ls='--', lw=.8); ax.axhline(-lvl, color=c, ls='--', lw=.8)
    ed = (res['eday'][y] | res['eday'][x]).loc[str(year)]
    for d in ed[ed].index:
        ax.axvline(d, color='purple', alpha=.35, lw=4)
    tr = res['trades'].query('pair == @pair')
    tr = tr[tr.entry_date.dt.year == year]
    ax.scatter(tr.entry_date, tr.entry_z, marker='^', color='black', zorder=5, label='entry')
    ax.scatter(tr.exit_date, tr.exit_z, marker='x', color='red', zorder=5, label='exit')
    ax.set_title(f'{pair} {year}: purple = earnings day (either bank)'); ax.legend(); plt.show()

first = res['picks'].iloc[0]
plot_pair_year(res, first.pair, int(first.year), CONFIG)

## 7. Does the earnings protection actually help? (ablation)

Same rules, switching each protection layer on and off. This is the table that answers "do earnings mess up the strategy?"

In [ ]:
variants = {
    'No earnings protection':      dict(use_blackout=False, use_neutralize=False, use_jump_filter=False),
    'Blackout only':               dict(use_blackout=True,  use_neutralize=False, use_jump_filter=False),
    'Blackout + neutralize':       dict(use_blackout=True,  use_neutralize=True,  use_jump_filter=False),
    'All three (base case)':       dict(use_blackout=True,  use_neutralize=True,  use_jump_filter=True),
}
abl = {}
for name, over in variants.items():
    r_ = run_backtest(px, earn, {**CONFIG, **over})
    rep = split_report(r_, bench, CONFIG)
    abl[name] = {'Dev Sharpe': rep.at['Sharpe', 'Development'], 'Holdout Sharpe': rep.at['Sharpe', 'Holdout'],
                 'Holdout CAGR': rep.at['CAGR', 'Holdout'], 'Holdout max DD': rep.at['Max drawdown', 'Holdout'],
                 'Holdout win rate': rep.at['Win rate', 'Holdout'] if 'Win rate' in rep.index else np.nan}
abl = pd.DataFrame(abl).T
abl

## 8. Robustness: does it only work at one setting?

Holdout Sharpe across entry thresholds and lookback windows, plus double costs. A real effect should look similar across nearby settings, not spike in one cell.

In [ ]:
grid = {}
for ze in [1.5, 2.0, 2.5]:
    for zw in [30, 60, 120]:
        r_ = run_backtest(px, earn, {**CONFIG, 'z_entry': ze, 'z_window': zw})
        grid[(ze, zw)] = split_report(r_, bench, CONFIG).at['Sharpe', 'Holdout']
heat = pd.Series(grid).unstack()
heat.index.name, heat.columns.name = 'z entry', 'z window (days)'

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(heat.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(heat.columns)), heat.columns); ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xlabel(heat.columns.name); ax.set_ylabel(heat.index.name)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f'{heat.values[i, j]:.2f}', ha='center', va='center')
ax.set_title('Holdout Sharpe (after costs)'); ax.grid(False); plt.colorbar(im); plt.show()

r2 = run_backtest(px, earn, {**CONFIG, 'cost_bps': CONFIG['cost_bps'] * 2})
print('Holdout Sharpe at 2x costs:', round(split_report(r2, bench, CONFIG).at['Sharpe', 'Holdout'], 2))

In [ ]:
# Save outputs for the write-up
summary.to_csv('summary.csv'); abl.to_csv('ablation.csv'); heat.to_csv('sensitivity.csv')
res['trades'].to_csv('trades.csv', index=False); res['picks'].to_csv('pairs_by_year.csv', index=False)
print('Saved: summary.csv, ablation.csv, sensitivity.csv, trades.csv, pairs_by_year.csv')

## 9. Report outline (≈8 pages)

1. **Executive summary** — question, one chart (growth of $1), answer in plain English
2. **Hypothesis & pre-registration** — paste section 0
3. **Data & method** — universe, walk-forward pair selection, signal, execution lag, costs
4. **Earnings handling** — the 3 layers + ablation table (section 7)
5. **Results** — development vs holdout table, annual returns, exit reasons, P&L by pair
6. **When it broke** — 2008, 2020, 2023 (US regional banks), bank-specific news; use the jump log and drawdown chart
7. **Robustness** — sensitivity heatmap, 2x costs
8. **Limitations** — daily closes, constant-weight approximation, borrow availability, multiple testing (15 pairs → ~1 false positive expected at p < 0.05), earnings-date coverage
9. **Conclusion** — what the evidence supports, and what would change your mind